# ⚖️ Aula 09 — Amostragem e Balanceamento de Classes

## 🎯 Como selecionar dados e lidar com classes desbalanceadas

**Disciplina:** ISW-039 — Mineração de Dados  
**Curso:** Desenvolvimento de Software Multiplataforma (DSM)  
**Ambiente:** Google Colab  
**Linguagem:** Python  
**Bibliotecas:** Pandas, NumPy, Matplotlib e Scikit-learn

---

## 🎯 Objetivos da aula

Ao final desta aula, você deverá ser capaz de:

- Compreender o conceito de amostragem;
- Diferenciar população e amostra;
- Conhecer técnicas de amostragem;
- Realizar amostragem aleatória com Python;
- Separar dados em treino e teste;
- Compreender o problema de classes desbalanceadas;
- Identificar uma classe minoritária;
- Aplicar **undersampling**;
- Aplicar **oversampling**;
- Comparar as distribuições antes e depois do balanceamento;
- Compreender os riscos de alterar artificialmente uma base;
- Preparar uma base para futuros modelos de classificação.

> **Projeto didático:** continuaremos utilizando o monitoramento de motores elétricos. Nesta aula, vamos criar uma variável `falha` para representar um cenário de manutenção preditiva.


# 🏭 1. O problema

Imagine que uma indústria possui **10.000 registros de motores**.

Desses registros:

```text
9.700 → operação normal
  300 → falha
```

Se um algoritmo simplesmente disser:

> "Todos os motores estão normais."

ele acertaria:

```text
9.700 / 10.000 = 97%
```

Mas o modelo seria completamente inútil para detectar falhas.

Esse é o problema do **desbalanceamento de classes**.

```text
Classe 0 → Normal → maioria
Classe 1 → Falha  → minoria
```

A acurácia sozinha pode esconder um modelo ruim.


# 👥 2. População × Amostra

### População

É o conjunto completo que queremos estudar.

Exemplo:

> Todos os registros de motores de uma indústria.

### Amostra

É uma parte da população utilizada para análise.

Exemplo:

> 2.000 registros selecionados entre 100.000 registros.

```text
POPULAÇÃO
████████████████████████████████████

          ↓ amostragem

AMOSTRA
████████
```

A amostra precisa representar adequadamente a população para que nossas conclusões sejam úteis.


# 🎲 3. Por que utilizar amostragem?

Imagine um banco com milhões de registros.

Nem sempre precisamos utilizar todos os dados durante uma análise exploratória.

A amostragem pode:

- reduzir custo computacional;
- acelerar experimentos;
- facilitar análises;
- permitir testes rápidos;
- ajudar na construção de conjuntos de treino e teste.

Mas existe um risco:

> Uma amostra mal escolhida pode produzir conclusões erradas.


# 💻 4. Preparando o ambiente

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

np.random.seed(42)

print("Ambiente preparado!")

# 🏭 5. Criando uma base industrial

Vamos criar uma base simulando leituras de motores.

Teremos:

- temperatura;
- vibração;
- corrente;
- tensão;
- RPM;
- falha.

A variável `falha` será nosso **rótulo (target)**.


In [ ]:
n = 1000

df = pd.DataFrame({
    "temperatura": np.random.normal(65, 8, n),
    "vibracao": np.random.normal(2.2, 0.7, n),
    "corrente": np.random.normal(13, 2, n),
    "tensao": np.random.normal(380, 4, n),
    "rpm": np.random.normal(1740, 15, n)
})

# Criando um cenário em que poucos registros representam falha
df["falha"] = 0

indices_falha = df[
    (df["temperatura"] > 78) &
    (df["vibracao"] > 3)
].index

df.loc[indices_falha, "falha"] = 1

df.head()

Vamos verificar quantas falhas foram geradas.


In [ ]:
df["falha"].value_counts()

Dependendo da distribuição aleatória, a quantidade pode variar.

Vamos visualizar.


In [ ]:
df["falha"].value_counts().sort_index().plot(kind="bar")
plt.title("Distribuição das Classes")
plt.xlabel("Classe")
plt.ylabel("Quantidade")
plt.xticks([0, 1], ["Normal", "Falha"], rotation=0)
plt.show()

# ⚠️ 6. Identificando o desbalanceamento

Vamos calcular a porcentagem de cada classe.


In [ ]:
proporcao = df["falha"].value_counts(normalize=True) * 100
proporcao

Imagine que encontramos:

```text
Normal → 95%
Falha  → 5%
```

Temos uma classe majoritária e uma classe minoritária.

Quanto maior a diferença entre as classes, maior a atenção necessária durante a preparação dos dados.


# 🎯 7. Amostragem aleatória

O Pandas permite selecionar uma amostra utilizando `sample()`.


In [ ]:
amostra = df.sample(n=100, random_state=42)

amostra.shape

Podemos também selecionar uma porcentagem da base.


In [ ]:
amostra_20 = df.sample(frac=0.20, random_state=42)

amostra_20.shape

O parâmetro `random_state` permite reproduzir o mesmo resultado.


# 🔬 8. Amostragem estratificada

Um problema da amostragem aleatória é que podemos selecionar uma amostra com uma proporção diferente de classes.

Quando existe uma variável importante, podemos utilizar **amostragem estratificada**.

Vamos utilizar a variável `falha`.


In [ ]:
amostra_estratificada, _ = train_test_split(
    df,
    test_size=0.80,
    stratify=df["falha"],
    random_state=42
)

print("Base original:")
print(df["falha"].value_counts(normalize=True))

print("\nAmostra estratificada:")
print(amostra_estratificada["falha"].value_counts(normalize=True))

A estratificação tenta preservar a proporção das classes.

Isso é especialmente importante quando a classe de interesse é rara.


# ✂️ 9. Divisão entre treino e teste

Antes de criar um modelo, normalmente dividimos os dados em:

```text
Treinamento → utilizado para aprender
Teste       → utilizado para avaliar
```

Exemplo:

```text
80% → treino
20% → teste
```


In [ ]:
X = df.drop("falha", axis=1)
y = df["falha"]

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Treino:", X_treino.shape)
print("Teste:", X_teste.shape)

Vamos conferir a distribuição.


In [ ]:
print("Treino:")
print(y_treino.value_counts(normalize=True))

print("\nTeste:")
print(y_teste.value_counts(normalize=True))

### ⚠️ Regra importante

Quando trabalhamos com classificação, a divisão treino/teste deve ser feita de forma cuidadosa.

E existe outro cuidado ainda mais importante:

> **O balanceamento deve ser realizado somente nos dados de treinamento.**

Se fizermos oversampling antes da divisão, podemos acabar colocando registros duplicados ou sintéticos relacionados no treino e no teste, causando **vazamento de dados (data leakage)**.


# ⚖️ 10. O que é balanceamento?

Balancear uma base significa modificar a distribuição das classes para reduzir uma diferença excessiva entre elas.

Existem duas técnicas básicas:

### Undersampling

Reduz a classe majoritária.

```text
Antes:

████████████████████  Normal
█                    Falha

Depois:

█████                Normal
█████                Falha
```

### Oversampling

Aumenta a classe minoritária.

```text
Antes:

████████████████████  Normal
█                    Falha

Depois:

████████████████████  Normal
██████████████████    Falha
```


# ✂️ 11. Undersampling

Vamos trabalhar somente com o conjunto de treinamento.

Primeiro vamos separar as classes.


In [ ]:
treino = X_treino.copy()
treino["falha"] = y_treino.values

classe_maioria = treino[treino["falha"] == 0]
classe_minoria = treino[treino["falha"] == 1]

print("Maioria:", len(classe_maioria))
print("Minoria:", len(classe_minoria))

Vamos reduzir a classe majoritária para o mesmo tamanho da minoritária.


In [ ]:
classe_maioria_reduzida = classe_maioria.sample(
    n=len(classe_minoria),
    random_state=42
)

treino_under = pd.concat([
    classe_maioria_reduzida,
    classe_minoria
]).sample(frac=1, random_state=42)

treino_under["falha"].value_counts()

Visualizando o resultado:


In [ ]:
treino_under["falha"].value_counts().sort_index().plot(kind="bar")
plt.title("Undersampling")
plt.xlabel("Classe")
plt.ylabel("Quantidade")
plt.xticks([0, 1], ["Normal", "Falha"], rotation=0)
plt.show()

### Vantagem

- reduz o tamanho da base;
- pode acelerar o treinamento.

### Desvantagem

- podemos descartar informações importantes da classe majoritária.


# ➕ 12. Oversampling

Agora vamos fazer o contrário.

Vamos aumentar a classe minoritária.

Uma forma simples é utilizar amostragem **com reposição** (`replace=True`).


In [ ]:
classe_minoria_aumentada = classe_minoria.sample(
    n=len(classe_maioria),
    replace=True,
    random_state=42
)

treino_over = pd.concat([
    classe_maioria,
    classe_minoria_aumentada
]).sample(frac=1, random_state=42)

treino_over["falha"].value_counts()

Visualizando:


In [ ]:
treino_over["falha"].value_counts().sort_index().plot(kind="bar")
plt.title("Oversampling")
plt.xlabel("Classe")
plt.ylabel("Quantidade")
plt.xticks([0, 1], ["Normal", "Falha"], rotation=0)
plt.show()

### Vantagem

- preserva todos os exemplos da classe majoritária;
- aumenta a representação da classe minoritária.

### Desvantagem

- pode duplicar exemplos;
- pode aumentar o risco de overfitting.


# 🧬 13. Oversampling sintético — SMOTE

Existe uma técnica bastante utilizada chamada **SMOTE — Synthetic Minority Over-sampling Technique**.

Em vez de simplesmente duplicar registros, ela cria novos exemplos sintéticos baseados em exemplos existentes da classe minoritária.

Vamos utilizar a implementação disponível no `imbalanced-learn`.


In [ ]:
!pip -q install imbalanced-learn

In [ ]:
from imblearn.over_sampling import SMOTE

Vamos aplicar SMOTE somente ao conjunto de treinamento.


In [ ]:
smote = SMOTE(random_state=42)

X_treino_smote, y_treino_smote = smote.fit_resample(
    X_treino,
    y_treino
)

print("Antes:")
print(y_treino.value_counts())

print("\nDepois:")
print(pd.Series(y_treino_smote).value_counts())

### O que aconteceu?

O SMOTE criou novos exemplos sintéticos da classe minoritária.

```text
Treino original
      ↓
SMOTE
      ↓
Treino balanceado
```

> O SMOTE será aprofundado posteriormente quando trabalharmos com modelos de classificação. Nesta aula o objetivo é compreender o problema e a lógica do balanceamento.


# 🔍 14. Comparando as técnicas

Vamos comparar:

```text
Original
Undersampling
Oversampling
SMOTE
```


In [ ]:
comparacao = pd.DataFrame({
    "Original": y_treino.value_counts(),
    "Undersampling": treino_under["falha"].value_counts(),
    "Oversampling": treino_over["falha"].value_counts(),
    "SMOTE": pd.Series(y_treino_smote).value_counts()
}).fillna(0).astype(int)

comparacao

Agora podemos observar visualmente.


In [ ]:
comparacao.plot(kind="bar", figsize=(9, 5))
plt.title("Comparação das Técnicas de Balanceamento")
plt.xlabel("Classe")
plt.ylabel("Quantidade")
plt.xticks(rotation=0)
plt.show()

# 🧠 15. Balanceamento não significa necessariamente 50/50

É comum começar aprendendo com a ideia de:

```text
50% classe 0
50% classe 1
```

Mas isso não é uma regra universal.

Dependendo do problema, podemos utilizar outras proporções.

O objetivo é criar uma distribuição que permita ao modelo aprender adequadamente sem destruir características importantes da população.

A decisão depende:

- do problema;
- da quantidade de dados;
- da classe minoritária;
- do algoritmo;
- do custo de falsos positivos;
- do custo de falsos negativos.


# 🚨 16. O problema do vazamento de dados

Observe o fluxo correto:

```text
BASE ORIGINAL
     ↓
Treino / Teste
     ↓
   ┌───────┐
   ↓       ↓
 TREINO   TESTE
   ↓
Balanceamento
   ↓
Modelo
```

Não devemos fazer:

```text
BASE ORIGINAL
     ↓
Balanceamento
     ↓
Treino / Teste
```

porque informações derivadas do conjunto completo podem acabar influenciando o teste.

O conjunto de teste deve permanecer o mais próximo possível de dados nunca vistos pelo modelo.


# 📝 17. Exercícios

## Exercício 1 — População e amostra

Explique, utilizando o contexto industrial:

1. qual seria a população;
2. qual seria uma amostra;
3. por que uma amostra poderia ser utilizada.


In [ ]:
# Sua resposta



## Exercício 2 — Amostragem

Selecione aleatoriamente 15% da base utilizando `sample()`.

Mostre o tamanho da amostra.


In [ ]:
# Sua resposta



## Exercício 3 — Distribuição

Calcule a quantidade e a porcentagem de cada classe de `falha`.


In [ ]:
# Sua resposta



## Exercício 4 — Amostragem estratificada

Crie uma amostra de 20% utilizando `train_test_split()` com `stratify`.

Compare a proporção das classes com a base original.


In [ ]:
# Sua resposta



## Exercício 5 — Treino e teste

Separe os dados em:

```text
80% treino
20% teste
```

Utilize estratificação pela variável `falha`.


In [ ]:
# Sua resposta



## Exercício 6 — Undersampling

Crie uma base de treinamento utilizando undersampling.

Mostre a distribuição final das classes.


In [ ]:
# Sua resposta



## Exercício 7 — Oversampling

Crie uma base de treinamento utilizando oversampling com reposição.

Compare a quantidade antes e depois.


In [ ]:
# Sua resposta



## Exercício 8 — SMOTE

Aplique SMOTE ao conjunto de treinamento.

Verifique a quantidade de registros antes e depois.


In [ ]:
# Sua resposta



## Exercício 9 — Comparação

Compare:

- base original;
- undersampling;
- oversampling;
- SMOTE.

Qual técnica preservou mais registros?
Qual técnica criou novos registros?


In [ ]:
# Sua resposta



## Exercício 10 — Decisão

Imagine que uma empresa deseja detectar falhas de motores.

O que seria pior?

**A)** indicar uma falha quando o motor está normal;

**B)** não identificar uma falha que realmente acontecerá.

Explique por que a resposta pode influenciar a estratégia de balanceamento e avaliação do modelo.


In [ ]:
# Sua resposta



# 🔎 18. Desafio — Preparando os dados para classificação

Monte um pipeline de preparação para uma futura classificação de falhas.

Seu processo deve conter:

```text
1. Separação de X e y
2. Treino e teste
3. Verificação das classes
4. Escolha de uma técnica de balanceamento
5. Aplicação somente no treino
6. Comparação antes/depois
7. Justificativa da escolha
```

No final, escreva:

> **Por que escolhi essa técnica para este problema?**


In [ ]:
# Desenvolva seu desafio aqui.



# 🚀 19. Aplicação no projeto do aluno

Agora pense no seu próprio projeto.

Você deverá verificar se existe uma variável que possa representar uma **classe ou categoria de interesse**.

Exemplos:

- fraude / não fraude;
- aprovado / reprovado;
- risco alto / baixo;
- defeito / sem defeito;
- cliente ativo / inativo;
- doença / ausência da doença;
- falha / operação normal.

Preencha:

| Item | Resposta |
|---|---|
| Qual é o problema de classificação? | ... |
| Qual seria a variável alvo? | ... |
| Existem classes? | ... |
| As classes estão equilibradas? | ... |
| Qual é a classe minoritária? | ... |
| Que risco existe no desbalanceamento? | ... |
| Qual técnica poderia ser utilizada? | ... |

> **Importante:** nem todo projeto precisará utilizar balanceamento. Primeiro devemos verificar se existe realmente um problema de desbalanceamento.


In [ ]:
# Análise do balanceamento do seu projeto



# 📌 20. Checklist da Aula

- [ ] Sei diferenciar população e amostra;
- [ ] Entendo por que utilizar amostragem;
- [ ] Sei realizar amostragem aleatória;
- [ ] Entendo amostragem estratificada;
- [ ] Sei separar treino e teste;
- [ ] Entendo o problema de classes desbalanceadas;
- [ ] Sei identificar classe majoritária e minoritária;
- [ ] Sei aplicar undersampling;
- [ ] Sei aplicar oversampling;
- [ ] Conheço o conceito de SMOTE;
- [ ] Entendo o risco de data leakage;
- [ ] Sei que o balanceamento deve ocorrer no conjunto de treinamento;
- [ ] Consigo analisar o balanceamento no meu projeto.

---

# 🎯 Conclusão

A sequência da disciplina está agora:

```text
Aula 3 → Pandas
Aula 4 → Limpeza
Aula 5 → ETL
Aula 6 → Web Scraping
Aula 7 → Banco de Dados + SQL
Aula 8 → Análise Exploratória
Aula 9 → Amostragem + Balanceamento
```

Até aqui aprendemos a:

```text
OBTER
  ↓
INTEGRAR
  ↓
LIMPAR
  ↓
EXPLORAR
  ↓
PREPARAR
```

Na próxima etapa começaremos a transformar os dados em informações visuais para facilitar a interpretação e comunicação dos resultados.

> 📊 **Próxima aula: Visualização de Dados**
